# FitPredictIsolationForestWrapper Basics

This tutorial introduces `FitPredictIsolationForestWrapper`, which extends `AbstractFitPredictWrapper` for sklearn-style `fit(X, y)` and `predict(X)` APIs. Isolation Forest is unsupervised; `y` is ignored at fit time.

## AbstractFitPredictWrapper Interface

```mermaid
flowchart TB
    subgraph Interface["AbstractFitPredictWrapper"]
        fit["fit(X, y)"]
        predict["predict(X)"]
    end
    
    subgraph Inputs["Inputs (torch.Tensor)"]
        X["X: (N, features)"]
        y["y: (N,) — ignored for IsolationForest"]
    end
    
    subgraph Outputs["Outputs"]
        pred["pred: (N, 1) or (N, 2) — 2D tensor"]
    end
    
    X --> fit
    y --> fit
    X --> predict
    fit --> predict
    predict --> pred
```

## Batch vs Full Predict

- **Full predict** (default): `predict(X)` processes all samples at once. Output is a single 2D tensor.
- **Batch predict** (`yield_strategy=True`): For large datasets, set `yield_strategy=True` and `yield_batch_size` to process in batches. Outputs are concatenated into one tensor.

In [ ]:
import numpy as np
import torch

from picid.model.estimators.isolation_forest.wrapper import (
    FitPredictIsolationForestWrapper,
)

# Create X (100, 5) and y (100,) — y is dummy for IsolationForest (unsupervised)
X = np.random.randn(100, 5).astype(np.float32)
y = np.zeros(100, dtype=np.float32)

X_t = torch.from_numpy(X)
y_t = torch.from_numpy(y)

model = FitPredictIsolationForestWrapper(task_type="anomaly_detection")
model.fit(X_t, y_t)

pred = model.predict(X_t)
print(f"pred shape: {pred.shape}")
assert pred.ndim == 2 and pred.shape[0] == 100
print("OK")

## Summary

- `fit(X, y)` and `predict(X)` expect **torch.Tensor** inputs; the wrapper converts to numpy internally for sklearn.
- Isolation Forest returns `(N, 2)` logits: [score_normal, score_anomaly].
- Use `yield_strategy=True` for batch-wise prediction on large datasets.